# 20｜視覺與多模態Transformer

把影像切成 patches，再以正規化向量比較圖文相似度。

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
import importlib.util, subprocess, sys
_needed = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])


In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.gcf().savefig(folder / f"{name}.svg", bbox_inches="tight")
    plt.show()

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")


## 課堂小實驗

In [ ]:
image = np.arange(16).reshape(4,4)
patches = np.stack([image[i:i+2,j:j+2].ravel() for i in (0,2) for j in (0,2)])
image_emb = np.array([[.9,.1], [.2,.8]])
text_emb = np.array([[1.,0.], [0.,1.], [.7,.7]])
labels = ['貓', '街景', '戶外動物']
norm = lambda x: x / np.linalg.norm(x, axis=1, keepdims=True)
sim = norm(image_emb) @ norm(text_emb).T
print('patch matrix:\n', patches)
pd.DataFrame(sim, index=['image_A','image_B'], columns=labels).round(3)

## 執行結果圖

In [ ]:
plt.imshow(sim, cmap='viridis', vmin=0, vmax=1)
plt.xticks(range(3), labels); plt.yticks(range(2), ['image_A','image_B'])
plt.colorbar(label='cosine similarity'); plt.title('Normalized image-text similarity')
for (i,j), value in np.ndenumerate(sim): plt.text(j, i, f'{value:.2f}', ha='center', va='center', color='white')
savefig('20_multimodal_demo')
report('multimodal', {'patches': patches.tolist(), 'similarity': sim.tolist()})

## 練習：改動一個參數

先預測結果，再修改程式中的一個參數並重跑；比較圖表或數值，說明差異。